# Rig tuning & colour QA

Interactive companion to the automated tests: grab frames, tune exposure/gain
**live**, and **eyeball colour** — the subjective thing HW-002 can't assert.

**Prerequisites**
- Run this under a kernel whose Python has **both `vision` and PySpin**. On the
  rig that's the interpreter with the Spinnaker SDK, e.g.
  `/usr/bin/python3.10 -m pip install -e ".[dev,notebook]"`, then launch Jupyter
  from there.
- A BlackFly S connected. Set `SERIAL` below (or leave `None` for device index 0).

Run the sections independently. Anything that grabs frames needs the camera
open first (`open_camera()`); the colour-comparison cell reopens per option and
closes at the end.

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt

# --- make the `vision` package importable from notebooks/ (src layout) ---
_d = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_d, "src", "vision")):
        break
    _d = os.path.dirname(_d)
_ROOT = _d
sys.path.insert(0, os.path.join(_ROOT, "src"))
sys.path.insert(0, _ROOT)

from vision.config_loader import load_camera_config
from vision.camera_types import PixelFormat, CameraStatus
from vision.image_ops import gray_world_balance
from vision.spinnaker_driver import SpinnakerCameraDriver
import PySpin  # confirms the SDK is importable in this kernel

print("repo root:", _ROOT)
print("vision + PySpin import OK")

In [ ]:
SERIAL = None                                    # e.g. "21512345"; None -> index 0
CONFIG = os.path.join(_ROOT, "config", "bfs_u3_16s2c.json")

In [ ]:
_drv = None

def open_camera(**overrides):
    # (Re)open the camera with config overrides; returns a streaming driver.
    global _drv
    close_camera()
    cfg = load_camera_config(CONFIG)
    if SERIAL:
        cfg.serial = SERIAL
    for k, v in overrides.items():
        setattr(cfg, k, v)
    _drv = SpinnakerCameraDriver(cfg)
    _drv.connect(); _drv.start_stream()
    return _drv

def close_camera():
    global _drv
    if _drv is not None:
        try:
            _drv.disconnect()
        finally:
            _drv = None

def grab(n=1, timeout=2.0):
    # Read n frames from the open camera; return the last (freshest).
    frame = None
    for _ in range(n):
        frame = _drv.read_frame(timeout=timeout)
    return frame

def to_rgb(frame):
    # Frames are BGR8 unless pixel_format is RGB8; matplotlib wants RGB.
    d = frame.data
    if d.ndim == 2:
        return d
    return d if frame.pixel_format == PixelFormat.RGB8 else d[..., ::-1]

def show(frame, title=""):
    plt.figure(figsize=(7, 5))
    plt.imshow(to_rgb(frame), cmap="gray" if frame.data.ndim == 2 else None)
    plt.title(title or f"frame {frame.frame_id}  {tuple(frame.data.shape)}")
    plt.axis("off"); plt.show()

## 1 · Grab a frame

In [ ]:
open_camera()                 # default config
show(grab(n=3), "default config")   # grab a few; show the freshest

## 2 · Live exposure / gain tuning
Change the values and **re-run this cell** — the writes go straight to the camera
nodes while streaming (no reconnect). `resulting_fps` is what the device can
actually sustain at these settings.

In [ ]:
_drv.set_config("exposure_us", 5000.0)   # microseconds (<= 1e6/fps)
_drv.set_config("gain_db",      0.0)     # dB (raise only if still dark)
f = grab(n=3)
print("resulting_fps:", _drv.get_health().get("resulting_fps"))
show(f, f"exp={_drv.get_config('exposure_us'):.0f}us  gain={_drv.get_config('gain_db'):.1f}dB")

## 3 · Compare white-balance / CCM options side by side
Each panel is a **real capture through the full pipeline** with that config
(reconnects per option). This is the subjective colour call HW-002 can't make.

In [ ]:
options = [
    ("none",               dict(white_balance="none")),
    ("gray_world",         dict(white_balance="gray_world")),
    ("ccm LED_4649K",      dict(white_balance="ccm", ccm_color_temp="LED_4649K")),
    ("ccm DAYLIGHT_5034K", dict(white_balance="ccm", ccm_color_temp="DAYLIGHT_5034K")),
]
shots = []
for label, ov in options:
    open_camera(**ov)
    shots.append((label, grab(n=3)))
close_camera()

fig, axes = plt.subplots(1, len(shots), figsize=(5 * len(shots), 5))
for ax, (label, fr) in zip(np.atleast_1d(axes), shots):
    ax.imshow(to_rgb(fr)); ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · Health + timing jitter
Grabs a burst and plots device-clock inter-frame intervals — the same signal the
acceptance battery scores, but visual.

In [ ]:
open_camera()
hw = []
for _ in range(100):
    fr = _drv.read_frame(timeout=2.0)
    if fr.hw_timestamp_ns is not None:
        hw.append(fr.hw_timestamp_ns)

h = _drv.get_health()
print("health:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in h.items()})
if len(hw) > 2:
    dt = np.diff(hw) / 1e6  # ns -> ms
    print(f"interframe: mean={dt.mean():.3f}ms  jitter(std)={dt.std():.3f}ms  max={dt.max():.3f}ms")
    plt.figure(figsize=(7, 3))
    plt.hist(dt, bins=30); plt.xlabel("inter-frame interval (ms)"); plt.ylabel("count")
    plt.title("timing jitter"); plt.show()
close_camera()

## 5 · Release the camera
Always run this when done (also runs on kernel restart).

In [ ]:
close_camera()
print("camera released")